# Lifecycle

This chapter is about the lifecycle of stream processing with Kafi Streams. It is split into three parts. 

First, we explain how to [set up and build](#setup_build) a processing topology, i.e., how to define the *topology* including *sources* and *sinks* and then how to build it.

Second, we describe the options how to [run and debug](#run_debug) the processing topology both using just the *TopologyNode* class and the *Streams* class (connects to Kafka).

The third and last section of this chapter is about how to [stop](#stop) the processing.


## Overview

[Preparation](#prep)

* [1 Set up and build](#setup_build)
  * [Set up](#setup)
    * [Sources](#sources)
      * [TopologyNode.source()](#topologynode_source)
      * [Streams.source()](#streams_source)
    * [Sinks](#sinks)
        * [TopologyNode.sink()](#topologynode_sink)
        * [Streams.sink()](#streams_sink)
    * [Build](#build)
      * [build()](#build-method)
      * [reset()](#reset)
    * [Sinks and build() under the covers](#sinks_and_build)
* [2 Run and debug](#run_debug)
  * [Run](#run)
    * [TopologyNode](#topologynode)
      * [push()](#push)
      * [latest()](#latest)
      * [process()](#process())
    * [Streams](#streams)
      * [start_streams()](#start_streams)
      * [streams()](#streams-method)
      * [streams_fun()](#streams_fun)
      * [threads()](#threads)
  * [Debug](#debug)
    * [peek()](#peek)
    * [topology()](#topology)
    * [mermaid()](#mermaid)
* [3 Stop](#stop)
  * [stop_fun()](#stop_fun)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
import sys
sys.path.insert(1, ".")
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator
click_generator = ClickGenerator()
debezium_click_generator = ClickGenerator(debezium_bool=True)
weights_click_generator = ClickGenerator(weights_bool=True)

click_source_str = "clicks"
sink_str = "sink"


---
<a id="setup_build"></a>
## 1 Set up and build

Stream processing with Kafi Streams starts with setting up a topology as e.g. shown in the [Quickstart](#quickstart.ipynb).

In broad strokes, a topology consists of three parts:
1. [Sources](#source)
2. The stream processing logic using the relational [operators](operators.ipynb) of Kafi Streams
3. [Sinks](#sinks)


<a id="sources"></a>
### Sources

You specify sources using the `source()` operator. `source()` differs slightly depending on whether you use the *TopologyNode* or the *Streams* class.


<a id="topologynode_source"></a>
#### TopologyNode.source()

When you use the *TopologyNode* class, the `source()` operator just takes one parameter: the name of the source topology node:
```python
@staticmethod
def source(source_str, **kwargs):
    """Create a named input source node.
    
    Args:
        source_str: name of the input source
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created source topology node"""
```


<a id="streams_source"></a>
##### Streams.source()

When you use the *Streams* class, the `source()` operator takes three parameters:
1. the Kafi *Storage* (e.g. a connection to Kafka cluster) of the source`storage`,
2. the name of the source `source_str`,
3. the name of the source topic `topic_str` (optional; defaults to `source_str`)

```python
@staticmethod
def source(storage, source_str, topic_str=None, **kwargs):
    """Create a source node backed by a storage topic.

    Args:
        storage: storage backend implementing consumer() (e.g. Kafka)
        source_str: name of the input source
        topic_str: topic name on storage; defaults to source_str
        **kwargs: passed to storage.consumer() at runtime
    Returns:
        tn: the newly created source topology node"""
```


<a id="sinks"></a>
### Sinks

You specify sinks using the `sink()` operator. `sink()` differs slightly depending on whether you use the *TopologyNode* or the *Streams* class.


<a id="topologynode_sink"></a>
#### TopologyNode.sink()

When you use the *TopologyNode* class, the `sink()` operator just takes one parameter: the name of the sink topology node:

```python
def sink(self, sink_str):
    """Mark this node as a named output sink.
    
    Args:
        sink_str: name of the output sink
    Returns:
        tn: the marked topologynode"""
```


<a id="streams_sink"></a>
##### Streams.sink()

As for `source()`, when you use the *Streams* class, the `sink()` operator takes three parameters:
1. the Kafi *Storage* (e.g. a connection to Kafka cluster) `storage` of the sink,
2. the name of the sink `sink_str`,
3. the name of the sink topic `topic_str` (optional; defaults to `sink_str`)

```python
def sink(self, storage, sink_str, topic_str=None, **kwargs):
    """Mark a topology node as a sink backed by a topic.

    Args:
        storage: storage backend implementing producer() (e.g. Kafka)
        sink_str: name of the output sink
        topic_str: topic name on storage; defaults to sink_str
        **kwargs: passed to storage.producer() at runtime
    Returns:
        tn: the marked topology node"""
```


<a id="build"></a>
### Build

After having set up the topology including its sources and sinks, you proceed to "build" it.


<a id="build-method"></a>
#### build()

As already shown e.g. in the [Quickstart](quickstart.ipynb), the method for building a topology is `build()`:
```python
@staticmethod
def build(*sink_tn_tuple):
    """Build: Merge the sinks and build the pydbsp circuit for the topology.
    
    Args:
        *sink_tn_tuple: one or more sink tn to build
    Returns:
        built_tn: the built topology node"""
```


<a id="reset"></a>
#### reset()

With `reset()`, you can rebuild the pydbsp circuit attached to a built topology node from scratch. This also clears all state:

```python
def reset(self):
    """Rebuild the circuit from scratch, clearing all state."""
```

<a id="sinks_and_build"></a>
### Sinks and build() under the covers

How does this look like under the covers?

In Kafi Streams, "building" a topology essentially consists of two steps:
1. Merge the sinks into one node. The resulting topology is `merged_tn`.
2. Wires the topology `merged_tn` up with the underlying pydbsp library.

Let's have a look at a very simple topology with two sink nodes:


In [2]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"]})
)

non_distinct_sink_str = "non_distinct"
non_distinct_tn = click_tn.sink(non_distinct_sink_str)

distinct_sink_str = "distinct"
distinct_tn = (
    click_tn
    .distinct()
    .sink(distinct_sink_str)
)

tn = Tn.build(non_distinct_tn, distinct_tn)


Now let's see the graphical Mermaid representations of the two sub topologies, starting with `non_distinct_tn`:

```mermaid
graph TD
cf4fd5e8-edbd-4345-ab97-61484d0a93e5[source_clicks] --> 9c663855-1dcd-4045-acbe-4bcf05daa21a[map_op]
```

Then `distinct_tn`:
```mermaid
graph TD
9c663855-1dcd-4045-acbe-4bcf05daa21a[map_op] --> 8a0d5f4c-ecd4-4069-bdb5-27a1d7004003[distinct_op]
cf4fd5e8-edbd-4345-ab97-61484d0a93e5[source_clicks] --> 9c663855-1dcd-4045-acbe-4bcf05daa21a[map_op]
```

And then the built topology node `tn`:
```mermaid
graph TD
b282680e-4971-4c63-8c96-2fc771f5cac9[map_op] --> fe2da5fc-9023-4af1-842d-9960e41c566c[distinct_op]
6c1e31d7-cd16-40b9-9891-7bd1e85ac37a[source_clicks] --> b282680e-4971-4c63-8c96-2fc771f5cac9[map_op]
86aded0b-6dcd-4d5a-8a22-e9f7ed6f9a77[sink_distinct] --> e5914f1b-4cc7-4221-85e3-4d2f8d039f71[merge_op]
b282680e-4971-4c63-8c96-2fc771f5cac9[map_op] --> 450a7bb5-a1ec-4c83-b63f-8e8bbacd9148[sink_non_distinct]
fe2da5fc-9023-4af1-842d-9960e41c566c[distinct_op] --> 86aded0b-6dcd-4d5a-8a22-e9f7ed6f9a77[sink_distinct]
450a7bb5-a1ec-4c83-b63f-8e8bbacd9148[sink_non_distinct] --> e5914f1b-4cc7-4221-85e3-4d2f8d039f71[merge_op]
```

What you can observe is that, technically, the `build()` step uses the `merge()` operator to merge the sinks into one final leaf node.


---
<a id="run_debug"></a>
## Run and debug

This is about how to run and debug a Kafi Streams topology.


<a id="run"></a>
### Run

You can either run a Kafi Streams topology using the [*TopologyNode*](#topologynode) or using the [*Streams*](#streams) class.

Using the *TopologyNode* class is a bit akin to using the Kafka Streams *TopologyTestDriver*. As this class is not connected to Kafka at all, you can use it e.g. to write unit tests or debug your application, or just to learn how Kafi Streams and the underlying pydbsp tick behind the covers.

A more low-level aspect of Kafi Streams during runtime is about the [pydbsp integration](#pydbsp). You don't have to know how this works to be able to use Kafi Streams, but for those who are interested in its inner workings, here you are.


<a id="topologynode"></a>
#### TopologyNode

If you use the *TopologyNode* class itself:
* you can manually push data into Kafi Streams ([`push()`](#push)),
* and manually trigger the processing and get the result the processing the latest data ([`latest()`](#latest)).

The `process()` method is just syntactic sugar for subsequently calling `push()` and `latest()`.

<a id="push"></a>
##### push()

The `push()` method pushes new input data to a built topology. Its sole parameter is a dictionary mapping source names to lists of new input data:

```python
def push(self, source_str_input_any_list_dict):
    """Feed new input records(/weights) into one or more named sources.
    
    Args:
        source_str_input_any_list_dict: a dictionary mapping source names to input lists to push"""
```



<a id="latest"></a>
##### latest()

The `latest()` method processes the input up to now and returns the resulting latest output:
```python
def latest(self, gc_bool=True):
    """Process the input up to now and return the resulting latest output, optionally garbage-collecting the state.
    
    Args:
        gc_bool: if True, garbage-collect the state after processing (default = True)
    Return:
        output_any: the latest output"""
```



<a id="process"></a>
##### process()

`process()` pushes input (with `push()`) and returns the latest output (with `latest()`) in one call:

```python
def process(self, source_str_input_any_list_dict, gc_bool=True):
    """Push input and return the latest output in one call.
    
    Args:
        source_str_input_any_list_dict: a dictionary mapping source names to input lists to push
        gc_bool: if True, garbage-collect the state after processing (default = True)
    Return:
        output_any: the latest output"""
```


<a id="streams"></a>
#### Streams

The *Streams* class is a wrapper around the *TopologyNode* class supporting Kafka-based sources and sinks.

`Streams` continuously consumes the source topics, pushes the data to pydbsp, gets the outputs and produces them to the sink topics.

In a separate chapter, we explain [fault tolerance](checkpointing.ipynb) with checkpointing, also determining the delivery guarantees of Kafi Streams.

<a id="start_streams"></a>
##### start_streams()

Start a Streams thread and get the `stop_fun: None -> None` to stop it:
```python
@staticmethod
def start_streams(built_tn, checkpoint_storage=None, checkpoint_topic_str=None, checkpoint_interval_float=default_checkpoint_interval_float, **kwargs):
    """Run streams() in a background thread; returns a function to stop it.

    Args:
        built_tn: built tn to run
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic_str: topic name used to store checkpoints
        checkpoint_interval_float: seconds between checkpoints
        **kwargs: passed through to streams()
    Returns:
        stop_fun: None -> None function to stop the Streams processing thread"""
```


<a id="streams-method"></a>
##### streams()

Synchronous sub method for `start_streams` - can e.g. be used to run Streams synchronously:
```python
@staticmethod 
def streams(built_tn, checkpoint_storage=None, checkpoint_topic_str=None, checkpoint_interval_float=default_checkpoint_interval_float, stop_event=None, **kwargs):
    """Build producers/consumers from the topology's sources/sinks and run streams_fun().

    Args:
        built_tn: built tn to run
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic_str: topic name used to store checkpoints
        checkpoint_interval_float: seconds between checkpoints
        stop_event: threading.Event that stops the loop once set
        **kwargs: passed through to storage.producer()/consumer()"""
```


<a id="streams_fun"></a>
##### streams_fun()

Sub method for `streams()`, actually the main method of Streams. Can be called directly e.g. if you'd like to set up the consumers/producers yourself. You can even use your own `foreach_fun` definition and e.g. call an API for each new sink output instead of producing to a sink Kafka topic:
```python
@staticmethod
def streams_fun(built_tn, sink_str_foreach_fun_finally_fun_tuple_dict, checkpoint_storage=None, checkpoint_topic_str=None, checkpoint_interval_float=default_checkpoint_interval_float, stop_event=None, **kwargs):
    """Main streams loop: consume, push through the topology, produce, checkpoint, repeat.

    Args:
        built_tn: built tn to run
        sink_str_foreach_fun_finally_fun_tuple_dict: dict, sink_str -> (produce_fun, close_fun)
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic_str: topic name used to store checkpoints
        checkpoint_interval_float: seconds between checkpoints
        stop_event: threading.Event that stops the loop once set
        **kwargs: extra options, e.g. group, step_fun, progress, chunk_size_bytes"""
```


<a id="threads"></a>
##### threads()

Helper to list the running Streams background threads:
```python
@staticmethod
def threads():
    """All currently running Streams background threads.
    
    Returns:
        streams_thread_list: the list of currently running Streams threads"""
```


<a id="debug"></a>
### Debug

In this section, we describe how Kafi Streams topologies and *Streams* threads can be debugged.

A common way to debug topologies is to use the [`peek()`](#peek) operator to display the output of operators within the topology.

Kafi Streams also offers two methods for visualizing topologies, i.e., [`topology()`](#topology) and [`mermaid()`](#mermaid).



<a id="peek"></a>
#### peek()

A common way to debug your topologies is to use the `peek()` operator. You can use it to print out intermediate calculations at operator-level.

Here is an example where we peek the input of a `map()` operator (selecting only the field `view_time` from the `value`) and its output.


In [3]:
sink_tn = Tn.source(click_source_str).peek("input").map(lambda r: r["value"]["view_time"]).peek("output").sink(sink_str)
#
tn = Tn.build(sink_tn)
#
input_m_list = click_generator.generate(10)
#
_ = tn.process({click_source_str: input_m_list})


input: {'key': None, 'value': {'customer_id': 85, 'view_time': 115, 'ts': 1787587355134}}
input: {'key': None, 'value': {'customer_id': 4, 'view_time': 35, 'ts': 1787587355234}}
input: {'key': None, 'value': {'customer_id': 90, 'view_time': 58, 'ts': 1787587355334}}
input: {'key': None, 'value': {'customer_id': 28, 'view_time': 98, 'ts': 1787587355434}}
input: {'key': None, 'value': {'customer_id': 16, 'view_time': 66, 'ts': 1787587355534}}
input: {'key': None, 'value': {'customer_id': 99, 'view_time': 120, 'ts': 1787587355634}}
input: {'key': None, 'value': {'customer_id': 25, 'view_time': 58, 'ts': 1787587355734}}
input: {'key': None, 'value': {'customer_id': 59, 'view_time': 37, 'ts': 1787587355834}}
input: {'key': None, 'value': {'customer_id': 13, 'view_time': 71, 'ts': 1787587355934}}
input: {'key': None, 'value': {'customer_id': 59, 'view_time': 17, 'ts': 1787587356034}}
output: 115
output: 35
output: 58
output: 98
output: 66
output: 120
output: 37
output: 71
output: 17


<a id="topology"></a>
##### topology()

`topology()` gives you a representation of the topology as a nested expression string:
```python
def topology(self, include_ids=False):
    """Render the graph as a nested expression string.
    
    Args:
        include_ids: if True, include node ids in the rendered graph
    Returns:
        str: the graph from this topology node as a nested expression string"""
```

Here is an example how to use `topology()`:

In [4]:
sink_tn = Tn.source(click_source_str).map(lambda r: r["value"]).distinct().sink(sink_str)
tn = Tn.build(sink_tn)
tn.topology()


'sink_sink(distinct_op(map_op(source_clicks)))'

<a id="mermaid"></a>
##### mermaid()

With `mermaid()` you can get a graphical representation of the topology in Mermaid format. This can e.g. be included in Markdown files:
```python
def mermaid(self, include_ids=False):
    """Render the graph as a Mermaid diagram.
    
    Args:
        include_ids: if True, include node ids in the rendered graph
    Returns:
        str: the graph from this topology node as a Mermaid diagram"""
```

Here is an example how to use `mermaid()`:

In [5]:
sink_tn = Tn.source(click_source_str).map(lambda r: r["value"]).distinct().sink(sink_str)
tn = Tn.build(sink_tn)
print(tn.mermaid())


```mermaid
graph TD
45760f75-c6e4-457c-ac05-8df675d21918[source_clicks] --> 5c7019bc-43ad-4d15-bd3b-a9ddfc8c2ccf[map_op]
d8dae68e-1ee0-4dd9-a301-530c6f6b28a4[distinct_op] --> 62886238-6a3a-46a9-ad84-203183874dae[sink_sink]
5c7019bc-43ad-4d15-bd3b-a9ddfc8c2ccf[map_op] --> d8dae68e-1ee0-4dd9-a301-530c6f6b28a4[distinct_op]
```


---
<a id="stop"></a>
## Stop

This is about running How can I run a Streams thread? Now this section is by far the shortest of all the three in this chapter ;:)

You simply call the `stop_fun` that you got returned from `start_streams`. `stop_fun` will safely stop the Streams thread. An example of this is already in the [Quickstart](#quickstart).